# 第 20 章互动实验：合成缺陷与扩散前向过程

比较“先插缺陷再成像”和几种“成像后再贴缺陷”的做法，把锐度、噪声、峰值三件事拆开看；再可视化 DDPM 的前向加噪，并量出每一步还剩多少信号。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "20_合成缺陷与生成模型"
ASSIGNMENT = "20-1_缺陷合成与扩散前向.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


## 实验 1：缺陷在成像链的哪一步插入

同一条划痕画五遍。第一种是先画到版图上，再一起经过光学模糊和噪声（成像前插入）；后四种都是在已经成像的良品图上再贴，它们和第一种的差别可以拆成三件事：

| 做法 | 边缘锐度 | 划痕上的噪声 | 峰值 |
|---|---|---|---|
| A 成像前插入 | 被光学模糊过 | 和背景一样 | 被模糊压低 |
| B 成像后替换成固定灰度 | 锐 | 没有 | 高 |
| C 同 B，但把噪声补回去 | 锐 | 有 | 高 |
| D 同 B，但把灰度降到 A 的峰值 | 锐 | 没有 | 和 A 一样 |
| E 掩膜先按光学模糊的 sigma 模糊再贴 | 和 A 接近 | 少一些 | 和 A 接近 |

B 是最常见的“贴纸”，它同时改了三件事，只看 A 和 B 分不出是哪一件让它显得假。C、D、E 各改回一件。先预测：C、D、E 里哪一张看起来最接近 A？再运行，对照打印出来的三列数字。

In [ ]:
def blur(img, sigma):
    fy = np.fft.fftfreq(img.shape[0])[:, None]; fx = np.fft.fftfreq(img.shape[1])[None, :]
    return np.real(np.fft.ifft2(np.fft.fft2(img) * np.exp(-2 * (np.pi * sigma) ** 2 * (fy**2 + fx**2))))

SIGMA, NOISE, GRAY = 1.5, 0.03, 0.95        # 光学模糊、噪声标准差、划痕的灰度
rng = np.random.default_rng(0)
y, x = np.mgrid[0:96, 0:96]
clean = 0.3 + 0.4 * ((x % 10) < 5)
m = attempt(mod.scratch_mask, (96, 96), (20.0, 15.0), (70.0, 80.0), 2.0, 1.0)
variants = None
if m is not None:
    try:
        noise = rng.normal(scale=NOISE, size=clean.shape)
        good0 = blur(clean, SIGMA)                           # 没有噪声的良品图
        a0 = blur(mod.blend(clean, GRAY, m), SIGMA)          # A：先画到版图上，再模糊
        core = m >= 1.0                                      # 划痕的中心线附近
        peak_a = float(np.mean(a0[core]))                    # A 在划痕中心的平均灰度
        b0 = mod.blend(good0, GRAY, m)
        d0 = mod.blend(good0, peak_a, m)
        e0 = mod.blend(good0, GRAY, blur(m, SIGMA).clip(0, 1))
        # 每项：(没有噪声的图, 带噪声的图)。B、D、E 是贴在带噪声的良品图上，所以掩膜盖住的地方噪声被盖掉了
        variants = {
            "A 成像前插入": (a0, a0 + noise),
            "B 成像后替换成固定灰度": (b0, mod.blend(good0 + noise, GRAY, m)),
            "C 同 B，补回噪声": (b0, b0 + noise),
            "D 同 B，灰度降到 A 的峰值": (d0, mod.blend(good0 + noise, peak_a, m)),
            "E 掩膜先模糊再贴": (e0, mod.blend(good0 + noise, GRAY, blur(m, SIGMA).clip(0, 1))),
        }
    except NotImplementedError as exc:
        print("先完成作业里剩下的函数：", exc)
if variants is not None:
    fig, ax = plt.subplots(2, 3, figsize=(13, 8.4))
    ax[0, 0].imshow(m, cmap="gray"); ax[0, 0].set_title("软掩码（标签）")
    for a, (name, (_, img)) in zip(ax.ravel()[1:], variants.items()):
        a.imshow(img, cmap="gray", vmin=0.1, vmax=1.05); a.set_title(name)
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.show()
    near = (blur(m, 3.0) > 0.02)                             # 划痕附近的一条带子，用来量边缘
    print("三件事各自的数字（越接近 A 那一行，这件事就越像真的成像）：")
    print("  边缘最大梯度  划痕上的噪声标准差  划痕中心的平均灰度   做法")
    for name, (img0, img) in variants.items():
        gy, gx = np.gradient(img0)
        print(f"  {np.hypot(gy, gx)[near].max():>10.3f}{np.std((img - img0)[core]):>16.3f}{np.mean(img0[core]):>18.3f}       {name}")
    print("读法：C 只把噪声改回来了，D 只把峰值改回来了，E 把边缘（连带峰值）改回来了。哪一张图看起来最不像贴纸？")
    print("这个单元只能告诉你每种做法改了图像的哪些统计量。哪一项真的影响检测器，要训练分类器来量：")
    print("讲义第 3.2 节的结果是，起作用的主要是锐度和随之而来的峰值，不是噪声。")

## DDPM 前向过程

预测：t=300 时图像还能认出条纹吗？

“认不认得出”要有个判断办法。下面的单元除了画图，还对每个 t 打印两个相关系数：

- **逐像素**：x_t 和 x_0 逐个像素比。它回答“只看一个像素，能不能分出这里原来是亮还是暗”。
- **按条纹平均**：先把每条条纹（5 列宽、96 行长）里的像素平均成一个数，再和 x_0 比。它回答“看一整条，能不能分出亮暗”，眼睛做的更接近这一种。

先写下你对 t=300 时这两个数的预测，再运行。两个数差多少，说明“某一步还剩多少信息”取决于你问的是多大的结构（讲义第 5.3.1 节和第 5.3.3 节）。

In [ ]:
betas = attempt(mod.linear_beta_schedule, 1000)
if betas is not None:
    try:
        abar = mod.alpha_bars(betas)
        x0 = (clean - 0.5) * 2
        noise = np.random.default_rng(1).normal(size=x0.shape)
        ts = [0, 50, 150, 300, 600, 999]
        stripe_mean = lambda img: img[:, :95].reshape(96, 19, 5).mean(axis=(0, 2))    # 前 19 条完整的条纹，每条 5 列
        fig, ax = plt.subplots(1, len(ts), figsize=(15, 3))
        rows = []
        for a, t in zip(ax, ts):
            xt = mod.q_sample(x0, t, abar, noise)
            a.imshow(xt, cmap="gray"); a.set_title(f"t={t}, alpha_bar={abar[t]:.3f}"); a.axis("off")
            rows.append((t, abar[t], np.corrcoef(xt.ravel(), x0.ravel())[0, 1], np.corrcoef(stripe_mean(xt), stripe_mean(x0))[0, 1]))
        plt.show()
        print("     t   alpha_bar   逐像素的相关系数   按条纹平均后的相关系数")
        for t, ab, c_pix, c_stripe in rows:
            print(f"  {t:>4}{ab:>12.4f}{c_pix:>14.3f}{c_stripe:>18.3f}")
        print("条纹取值是 -0.4 和 0.4。逐像素的相关系数按公式是 sqrt(ab * 0.16 / (ab * 0.16 + 1 - ab))，ab 是 alpha_bar，可以自己代一个 t 核对；")
        print("按条纹平均后的那一列只有 19 条条纹，换一份噪声会有波动。")
        plt.plot(np.sqrt(abar), label="信号系数 sqrt(alpha_bar)"); plt.plot(np.sqrt(1 - abar), label="噪声系数 sqrt(1 - alpha_bar)")
        plt.xlabel("t"); plt.legend(); plt.show()
    except NotImplementedError as exc:
        print("先完成作业里剩下的函数：", exc)

## VAE 重参数化：采样的梯度从哪里来

In [ ]:
z = attempt(mod.reparameterize, np.zeros(5), np.log(np.full(5, 4.0)), np.ones(5))
if z is not None:
    print("mu=0、方差 4、eps=1 时 z =", z, "（dz/dmu = 1，dz/dlogvar = 0.5·sigma·eps，都和 eps 的采样无关）")

## 破坏实验 2 的起步代码：只用一种缺陷调阈值

讲义第 11 节的破坏实验 2：用第 24 章的 `数据生成.py`，只在带划痕的 die 上调一个差值阈值，再用这个阈值去测颗粒和图形缺失，检出率差多少？

下面的单元把数据准备好：借用第 24 章生成器里的版图和三种缺陷（划痕、颗粒、图形缺失），每张图放一个缺陷，经过模糊和噪声，再给每张图算一个分数（和无噪声参考图的差，先做 3×3 平均，再取绝对值的最大值）。为了让这个实验只看“缺陷种类”这一件事，这里没有加第 24 章的随机错位和亮度漂移，那两件事留给第 24 章。这个单元不依赖本章作业。

剩下的三步由你来写，单元末尾有提示。

In [ ]:
def load_ch24_generator():
    path = COURSE_ROOT / "24_综合项目_晶圆检测流水线" / "数据生成.py"
    if not path.exists():
        print("跳过：没有找到", path)
        return None
    spec = importlib.util.spec_from_file_location("ch24_datagen", path)
    gen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(gen)
    if not all(hasattr(gen, name) for name in ("make_layout", "_add_defect", "_gaussian_blur")):
        print("跳过：第 24 章的 数据生成.py 里没有找到 make_layout、_add_defect、_gaussian_blur，可能已经改过名字。")
        return None
    return gen

def box3(img):
    """3x3 平均，边界用复制填充。"""
    p = np.pad(img, 1, mode="edge")
    return sum(p[dy:dy + img.shape[0], dx:dx + img.shape[1]] for dy in range(3) for dx in range(3)) / 9

gen = load_ch24_generator()
kind_scores = None
if gen is not None:
    layout = gen.make_layout()
    reference = gen._gaussian_blur(layout, 1.0)               # 无噪声的参考图
    rng24 = np.random.default_rng(24)

    def make_die(kind=None):
        """一张 die：kind 为 None 是良品，否则放一个指定种类的缺陷（先加缺陷，再模糊，再加噪声）。"""
        img = layout.copy()
        if kind is not None:
            cy, cx = rng24.uniform(12, layout.shape[0] - 12, size=2)
            gen._add_defect(img, kind, cy, cx, rng24)
        return gen._gaussian_blur(img, 1.0) + rng24.normal(scale=0.02, size=img.shape)

    score = lambda img: float(np.abs(box3(img - reference)).max())
    kind_scores = {name: np.array([score(make_die(kind)) for _ in range(200)])
                   for name, kind in (("良品", None), ("划痕", "scratch"), ("颗粒", "particle"), ("图形缺失", "missing"))}
    fig, ax = plt.subplots(figsize=(9, 3.6))
    bins = np.linspace(0, max(v.max() for v in kind_scores.values()), 60)
    for name, v in kind_scores.items():
        ax.hist(v, bins=bins, alpha=0.55, label=name)
        print(f"{name}：200 张，分数的最小值 {v.min():.3f}，中位数 {np.median(v):.3f}，最大值 {v.max():.3f}")
    ax.set_xlabel("分数（和参考图的差，3x3 平均后取绝对值的最大值）"); ax.set_ylabel("张数"); ax.legend(); plt.show()
    print("接下来由你来写：")
    print("  1. 只看 kind_scores[\"划痕\"]，选一个阈值，让划痕的漏检率不超过 5%（阈值越大过检越少）。")
    print("  2. 用这个阈值统计良品的过检率，以及颗粒、图形缺失的检出率。")
    print("  3. 再只用“图形缺失”调一次阈值，良品的过检率变成多少？哪一种缺陷最容易被“只用一种调参”坑到？")
    print("  提示：作业第 19 章的 threshold_at_escape 做的就是第 1 步；没做第 19 章就用 np.quantile(划痕分数, 0.05)。")

## 破坏实验与记录

- 把软掩码的 softness 设得很小（接近硬边），再看实验 1 的 A 和 B 两张图：哪一张变了，哪一张几乎没变？
- 完成上面“破坏实验 2 的起步代码”后面的三步：只用划痕调阈值，再测颗粒和图形缺失，检出率差多少？
- 把 `linear_beta_schedule` 的步数改成 100、β 范围不变，重跑 DDPM 那个单元（`ts` 也要改），看最后一步的两个相关系数。
- 记录到 `实验记录.md`。